# ViGil — Quad-Modal Malware Detection
**Architecture:** HGT (CPG) + ResNet-50 (image) + RansomFormer (bytes + API) → Bayesian Neural Network

**Input:** Pre-extracted `.feat.pt` files (no PE binaries needed)

**Output:** `vigil_deploy.zip` — contains all model weights + standalone `predict.py`

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import os, sys, shutil
from pathlib import Path

# Copy source code from Kaggle dataset to writable working dir
INPUT_DIR   = Path('/kaggle/input/vigil-features')   # <-- your Kaggle dataset name
WORK_DIR    = Path('/kaggle/working/vigil')
FEAT_DIR    = INPUT_DIR                              # .feat.pt files are read directly
CKPT_DIR    = WORK_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Copy uir source code if uploaded separately
SRC_INPUT = Path('/kaggle/input/vigil-src')          # optional: uir source zip
if SRC_INPUT.exists():
    if not WORK_DIR.exists():
        shutil.copytree(str(SRC_INPUT), str(WORK_DIR))
    sys.path.insert(0, str(WORK_DIR))

print('torch version:', __import__('torch').__version__)
print('CUDA available:', __import__('torch').cuda.is_available())
print('Feature files:', len(list(FEAT_DIR.rglob('*.feat.pt'))))

In [ ]:
# ── Cell 2: Install missing deps ──────────────────────────────────────────────
!pip install -q pydantic tqdm

In [ ]:
# ── Cell 3: Inline model definitions (no uir package required) ────────────────
import math, torch, torch.nn as nn, torch.nn.functional as F
from typing import Tuple

# ---- Bayesian Linear ----
class BayesianLinear(nn.Module):
    def __init__(self, in_f, out_f, prior=0.1):
        super().__init__()
        self.prior = prior
        self.weight_mu  = nn.Parameter(torch.Tensor(out_f, in_f))
        self.weight_rho = nn.Parameter(torch.Tensor(out_f, in_f))
        self.bias_mu    = nn.Parameter(torch.Tensor(out_f))
        self.bias_rho   = nn.Parameter(torch.Tensor(out_f))
        nn.init.kaiming_uniform_(self.weight_mu, a=math.sqrt(5))
        nn.init.constant_(self.weight_rho, -3.0)
        bound = 1/math.sqrt(in_f)
        nn.init.uniform_(self.bias_mu, -bound, bound)
        nn.init.constant_(self.bias_rho, -3.0)
    def forward(self, x, sample=True):
        if self.training or sample:
            ws=torch.log1p(torch.exp(self.weight_rho))
            w =self.weight_mu+ws*torch.randn_like(self.weight_mu)
            bs=torch.log1p(torch.exp(self.bias_rho))
            b =self.bias_mu+bs*torch.randn_like(self.bias_mu)
        else: w,b=self.weight_mu,self.bias_mu
        return F.linear(x,w,b)
    def kl(self):
        ws=torch.log1p(torch.exp(self.weight_rho))
        bs=torch.log1p(torch.exp(self.bias_rho))
        kw=0.5*(2*torch.log(self.prior/(ws+1e-8))+(ws**2+self.weight_mu**2)/self.prior**2-1).sum()
        kb=0.5*(2*torch.log(self.prior/(bs+1e-8))+(bs**2+self.bias_mu**2)/self.prior**2-1).sum()
        return kw+kb

class BayesianClassifier(nn.Module):
    def __init__(self, in_f, n_cls=2, hidden=256):
        super().__init__()
        self.fc1=BayesianLinear(in_f,hidden); self.act=nn.GELU()
        self.fc2=BayesianLinear(hidden,n_cls)
    def forward(self,x,sample=True): return self.fc2(self.act(self.fc1(x,sample)),sample)
    def kl_divergence(self): return self.fc1.kl()+self.fc2.kl()

# ---- ResNet-50 Extractor ----
import torchvision.models as tv_models
class ResNetFeatureExtractor(nn.Module):
    OUT_DIM=384
    def __init__(self, pretrained=True):
        super().__init__()
        try:
            from torchvision.models import ResNet50_Weights
            bb=tv_models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        except Exception:
            bb=tv_models.resnet50(pretrained=pretrained)
        self.conv1=bb.conv1;self.bn1=bb.bn1;self.relu=bb.relu;self.maxpool=bb.maxpool
        self.layer1=bb.layer1;self.layer2=bb.layer2;self.layer3=bb.layer3;self.layer4=bb.layer4
        self.avgpool=nn.AdaptiveAvgPool2d((1,1));self.drop=nn.Dropout(0.2)
        self.proj=nn.Sequential(nn.Linear(2048,self.OUT_DIM),nn.LayerNorm(self.OUT_DIM),nn.GELU())
        for p in list(self.conv1.parameters())+list(self.bn1.parameters())+\
                 list(self.layer1.parameters())+list(self.layer2.parameters())+\
                 list(self.layer3.parameters()): p.requires_grad=False
    def forward(self,x):
        x=self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x=self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.proj(self.drop(self.avgpool(x).flatten(1)))

# ---- RansomFormer ----
class ByteEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1=nn.Sequential(nn.Conv1d(1,64,5,padding=2),nn.BatchNorm1d(64),nn.ReLU(),nn.MaxPool1d(2))
        self.c2=nn.Sequential(nn.Conv1d(64,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.MaxPool1d(2))
        self.pool=nn.AdaptiveAvgPool1d(64);self.fc=nn.Linear(128*64,256);self.drop=nn.Dropout(0.2)
    def forward(self,x): return self.fc(self.drop(self.pool(self.c2(self.c1(x))).flatten(1)))

class APIEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb=nn.Embedding(4096,256,padding_idx=0)
        enc=nn.TransformerEncoderLayer(256,8,512,0.2,'relu',batch_first=True)
        self.tr=nn.TransformerEncoder(enc,num_layers=8);self.drop=nn.Dropout(0.2)
    def forward(self,t):
        pm=(t==0).clone();pm[:,0]=False
        x=self.drop(self.emb(t));x=self.tr(x,src_key_padding_mask=pm)
        op=(~(t==0)).unsqueeze(-1).float()
        return (x*op).sum(1)/op.sum(1).clamp(min=1)

class RansomFormerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.be=ByteEncoder();self.ae=APIEncoder()
        self.attn=nn.MultiheadAttention(256,8,dropout=0.2,batch_first=True)
        self.norm=nn.LayerNorm(256);self.drop=nn.Dropout(0.2)
        self.qp=nn.Linear(256,256);self.kp=nn.Linear(256,256);self.vp=nn.Linear(256,256)
        self.out=nn.Sequential(nn.Linear(256,256),nn.LayerNorm(256),nn.GELU())
    def forward(self,b,a):
        bf=self.be(b);af=self.ae(a)
        Q=self.qp(bf).unsqueeze(1);K=self.kp(af).unsqueeze(1);V=self.vp(af).unsqueeze(1)
        ao,_=self.attn(Q,K,V);ao=ao.squeeze(1)
        return self.out(self.norm(bf+self.drop(ao)))

# ---- HGT ----
class HGTLayer(nn.Module):
    def __init__(self,h,heads):
        super().__init__()
        self.heads=heads;self.d=h//heads
        self.WQ=nn.ModuleList([nn.Linear(h,self.d) for _ in range(heads)])
        self.WK=nn.ModuleList([nn.Linear(h,self.d) for _ in range(heads)])
        self.WV=nn.ModuleList([nn.Linear(h,self.d) for _ in range(heads)])
        self.out=nn.Linear(h,h);self.norm=nn.LayerNorm(h)
    def forward(self,x,ei,*_):
        if ei.size(1)==0: return x
        s,d=ei[0],ei[1];outs=[]
        for h in range(self.heads):
            q=self.WQ[h](x)[d];k=self.WK[h](x)[s];v=self.WV[h](x)[s]
            a=torch.softmax((q*k).sum(-1,keepdim=True)/math.sqrt(self.d),0)
            agg=torch.zeros(x.size(0),self.d,device=x.device)
            agg.scatter_add_(0,d.unsqueeze(-1).expand_as(v*a),v*a);outs.append(agg)
        return self.norm(x+self.out(torch.cat(outs,-1)))

class HGT(nn.Module):
    def __init__(self,in_d,hidden,layers,heads):
        super().__init__()
        self.proj=nn.Linear(in_d,hidden)
        self.layers_=nn.ModuleList([HGTLayer(hidden,heads) for _ in range(layers)])
        self.pw=nn.Linear(hidden,1);self.out=nn.Linear(hidden,hidden*2)
    def get_graph_embedding(self,x,ei,nt,et,batch):
        x=F.gelu(self.proj(x))
        for l in self.layers_: x=l(x,ei,nt,et)
        w=torch.softmax(self.pw(x),0)
        B=int(batch.max().item())+1
        p=torch.zeros(B,x.size(1),device=x.device)
        p.scatter_add_(0,batch.unsqueeze(-1).expand_as(x*w),x*w)
        return self.out(p)

class JointMalwareModel(nn.Module):
    def __init__(self,hgt,resnet,rf,bnn):
        super().__init__()
        self.hgt=hgt;self.resnet=resnet;self.ransomformer=rf;self.bnn=bnn
    def forward(self,x,ei,nt,et,bi,imgs,pb,at,sample=True):
        g=self.hgt.get_graph_embedding(x,ei,nt,et,bi)
        i=self.resnet(imgs);r=self.ransomformer(pb,at)
        return self.bnn(torch.cat([g,i,r],-1),sample)
    @torch.no_grad()
    def predict_with_confidence(self,x,ei,nt,et,bi,imgs,pb,at,n=20):
        self.eval()
        probs=[torch.softmax(self.forward(x,ei,nt,et,bi,imgs,pb,at,True),-1) for _ in range(n)]
        probs=torch.stack(probs);mp=probs.mean(0);pred=mp.argmax(-1)
        conf=mp[torch.arange(pred.size(0)),pred]
        var=probs[:,torch.arange(pred.size(0)),pred].var(0)
        return pred,conf,var

print('✓ Model classes defined')

In [ ]:
# ── Cell 4: Dataset (loads .feat.pt files) ────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

class PreExtractedDataset(Dataset):
    def __init__(self, feat_dir):
        self.files = sorted(Path(feat_dir).rglob('*.feat.pt'))
        print(f'Found {len(self.files)} samples')
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        feat = torch.load(self.files[idx], map_location='cpu')
        return {
            'x':          feat.get('x',          torch.zeros(1,320)),
            'edge_index': feat.get('edge_index',  torch.zeros(2,0,dtype=torch.long)),
            'node_types': feat.get('node_types',  torch.zeros(1,dtype=torch.long)),
            'edge_types': feat.get('edge_types',  torch.zeros(0,dtype=torch.long)),
            'image':      feat.get('image',       torch.zeros(3,224,224)),
            'pe_bytes':   feat.get('pe_bytes',    torch.zeros(1,1024)),
            'api_tokens': feat.get('api_tokens',  torch.zeros(256,dtype=torch.long)),
            'label':      torch.tensor(int(feat.get('label',0)), dtype=torch.long),
        }

def collate(batch):
    xs=[]; eis=[]; nts=[]; ets=[]; imgs=[]; pbs=[]; ats=[]; ys=[]
    batches=[]; offset=0
    for i,b in enumerate(batch):
        n=b['x'].size(0); xs.append(b['x']); nts.append(b['node_types'])
        if b['edge_index'].size(1)>0:
            eis.append(b['edge_index']+offset); ets.append(b['edge_types'])
        imgs.append(b['image']); pbs.append(b['pe_bytes'])
        ats.append(b['api_tokens']); ys.append(b['label'].unsqueeze(0))
        batches.append(torch.full((n,),i,dtype=torch.long)); offset+=n
    from torch import cat, stack, zeros
    batch_idx=cat(batches)
    ei=cat(eis,1) if eis else zeros(2,0,dtype=torch.long)
    et=cat(ets)   if ets  else zeros(0,dtype=torch.long)
    return (cat(xs), ei, cat(nts), et, batch_idx,
            stack(imgs), stack(pbs), stack(ats), cat(ys).squeeze())

dataset = PreExtractedDataset(FEAT_DIR)
print(f'Dataset: {len(dataset)} samples')

In [ ]:
# ── Cell 5: Train/Val/Test split & DataLoaders ────────────────────────────────
from sklearn.model_selection import train_test_split

labels = []
for f in dataset.files:
    feat = torch.load(f, map_location='cpu')
    labels.append(int(feat.get('label', 0)))

indices = list(range(len(dataset)))
tv_idx, te_idx = train_test_split(indices, test_size=0.10, stratify=labels, random_state=42)
tv_lbl = [labels[i] for i in tv_idx]
tr_idx, va_idx = train_test_split(tv_idx, test_size=0.111, stratify=tv_lbl, random_state=42)

from torch.utils.data import Subset
BATCH = 32
train_dl = DataLoader(Subset(dataset,tr_idx), BATCH, shuffle=True,  collate_fn=collate, num_workers=2)
val_dl   = DataLoader(Subset(dataset,va_idx), BATCH, shuffle=False, collate_fn=collate, num_workers=2)
test_dl  = DataLoader(Subset(dataset,te_idx), BATCH, shuffle=False, collate_fn=collate, num_workers=2)

print(f'Train: {len(tr_idx)}  Val: {len(va_idx)}  Test: {len(te_idx)}')

In [ ]:
# ── Cell 6: Build model ───────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')

HIDDEN=256; LAYERS=4; HEADS=8; IN_DIM=320; FUSED=1152; N_CLS=2

hgt_model    = HGT(IN_DIM, HIDDEN, LAYERS, HEADS)
resnet_model = ResNetFeatureExtractor(pretrained=True)
rf_model     = RansomFormerEncoder()
bnn_model    = BayesianClassifier(FUSED, N_CLS, HIDDEN)
model        = JointMalwareModel(hgt_model, resnet_model, rf_model, bnn_model).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {trainable:,} / {total:,} trainable ({100*trainable/total:.1f}%)')

In [ ]:
# ── Cell 7: Training loop ─────────────────────────────────────────────────────
from sklearn.metrics import f1_score
import numpy as np

EPOCHS    = 50
LR        = 1e-4
KL_WEIGHT = 1e-4

optimizer  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
criterion  = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

best_f1   = 0.0
best_path = CKPT_DIR / 'best_joint_model.pt'

for epoch in range(1, EPOCHS+1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train(); tl=tc=0
    for x,ei,nt,et,bi,imgs,pb,at,y in train_dl:
        x=x.to(DEVICE);ei=ei.to(DEVICE);nt=nt.to(DEVICE);et=et.to(DEVICE)
        bi=bi.to(DEVICE);imgs=imgs.to(DEVICE);pb=pb.to(DEVICE);at=at.to(DEVICE);y=y.to(DEVICE)
        optimizer.zero_grad()
        logits=model(x,ei,nt,et,bi,imgs,pb,at,sample=True)
        loss=criterion(logits,y)+KL_WEIGHT*model.bnn.kl_divergence()
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        tl+=loss.item(); tc+=(logits.argmax(-1)==y).sum().item()
    train_acc=tc/len(tr_idx)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval(); preds_all=[]; labels_all=[]; vl=0
    with torch.no_grad():
        for x,ei,nt,et,bi,imgs,pb,at,y in val_dl:
            x=x.to(DEVICE);ei=ei.to(DEVICE);nt=nt.to(DEVICE);et=et.to(DEVICE)
            bi=bi.to(DEVICE);imgs=imgs.to(DEVICE);pb=pb.to(DEVICE);at=at.to(DEVICE);y=y.to(DEVICE)
            logits=model(x,ei,nt,et,bi,imgs,pb,at,sample=False)
            vl+=criterion(logits,y).item()
            preds_all.extend(logits.argmax(-1).cpu().tolist())
            labels_all.extend(y.cpu().tolist())
    val_f1=f1_score(labels_all,preds_all,average='weighted')
    val_acc=np.mean(np.array(preds_all)==np.array(labels_all))

    print(f'Epoch {epoch:3d}/{EPOCHS} | '
          f'Train Acc: {train_acc:.4f} | '
          f'Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}'
          + (' ← BEST' if val_f1>best_f1 else ''))

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_f1': val_f1}, best_path)

print(f'\n✅ Training complete. Best Val F1: {best_f1:.4f}')
print(f'   Best checkpoint: {best_path}')

In [ ]:
# ── Cell 8: Test set evaluation ───────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

# Load best weights
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

all_preds=[]; all_labels=[]
with torch.no_grad():
    for x,ei,nt,et,bi,imgs,pb,at,y in test_dl:
        x=x.to(DEVICE);ei=ei.to(DEVICE);nt=nt.to(DEVICE);et=et.to(DEVICE)
        bi=bi.to(DEVICE);imgs=imgs.to(DEVICE);pb=pb.to(DEVICE);at=at.to(DEVICE);y=y.to(DEVICE)
        logits=model(x,ei,nt,et,bi,imgs,pb,at,sample=False)
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(y.cpu().tolist())

print(classification_report(all_labels, all_preds, target_names=['Benign','Malware']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign','Malware'], yticklabels=['Benign','Malware'])
plt.title('Confusion Matrix — Test Set'); plt.tight_layout(); plt.savefig('/kaggle/working/confusion_matrix.png')
plt.show()

In [ ]:
# ── Cell 9: Export vigil_deploy.zip ──────────────────────────────────────────
import zipfile, json
from pathlib import Path

DEPLOY_ZIP  = Path('/kaggle/working/vigil_deploy.zip')
BEST_CKPT   = best_path

MODEL_CONFIG = {
    'embedding_dim': 320, 'hidden_dim': 256, 'num_heads': 8, 'num_layers': 4,
    'num_classes': 2, 'fused_dim': 1152, 'byte_seq_len': 1024,
    'max_apis': 256, 'api_vocab_size': 4096,
    'label_map': {'0': 'BENIGN', '1': 'MALWARE'},
    'architecture': 'HGT(512) + ResNet50(384) + RansomFormer(256) → BNN',
}

README = """# ViGil Deployment Package
## Requirements
pip install torch torchvision numpy pillow

## Predict
python vigil_deploy/predict.py --model vigil_deploy/models/joint_model.pt --file suspicious.exe
"""

# Inline predict.py (copy of the notebook Cell 3 + predict logic)
PREDICT_PY_URL = '/kaggle/input/vigil-src/predict.py'   # if uploaded

with zipfile.ZipFile(DEPLOY_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(BEST_CKPT, 'vigil_deploy/models/joint_model.pt')
    zf.writestr('vigil_deploy/model_config.json', json.dumps(MODEL_CONFIG, indent=2))
    zf.writestr('vigil_deploy/README.md', README)
    if Path(PREDICT_PY_URL).exists():
        zf.write(PREDICT_PY_URL, 'vigil_deploy/predict.py')

size_mb = DEPLOY_ZIP.stat().st_size / 1e6
print(f'\n📦  vigil_deploy.zip created: {DEPLOY_ZIP}  ({size_mb:.1f} MB)')
print('    Download from the Kaggle Output tab.')